In [0]:
from pyspark.sql.functions import *

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS world_cup_2026.gold")

In [0]:
df_fact = spark.table("world_cup_2026.silver.fact_matches")

# Unificamos estadísticas cuando actúan como local
df_home = df_fact.select(
    col("home_team").alias("team"),
    col("home_goals").alias("goals"),
    col("home_xg").alias("xg"),
    col("possession_home").alias("possession")
)

# Unificamos estadísticas cuando actúan como visitante
df_away = df_fact.select(
    col("away_team").alias("team"),
    col("away_goals").alias("goals"),
    col("away_xg").alias("xg"),
    col("possession_away").alias("possession")
)

# Unimos ambos DataFrames (Unpivot/Union de partidos)
df_all_teams = df_home.union(df_away)

In [0]:
# 2. Agregamos las métricas a nivel de País
df_gold_performance = (df_all_teams
    .groupBy("team")
    .agg(
        count("*").alias("total_matches"),
        sum("goals").alias("total_goals_scored"),
        avg("goals").alias("avg_goals_per_match"),
        sum("xg").alias("total_expected_goals_xg"),
        avg("possession").alias("avg_possession_pct")
    )
    .sort(col("total_goals_scored").desc())
)

In [0]:
# 3. Guardamos en la capa Gold
(df_gold_performance.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("world_cup_2026.gold.team_historical_performance"))

print("¡Tabla Gold analítica lista para Dashboards!")